# Cloud dataset intake

> **Do not run this notebook just to inspect archive contents.** It downloads and fully extracts the media and may take a long time. A central-directory inventory has already been checked remotely and is recorded in the dataset cards. Run this notebook only when you are ready to materialize the media in Colab for full-byte/CRC verification and downstream processing.

Run this notebook in Google Colab. It verifies both ZIP records in the supplied Drive folder, downloads both Drive ZIPs and extracts the Comprehensive dataset's nested ZIP from its Drive wrapper into temporary Colab storage, safely extracts them, and writes inventory CSVs plus a summary into intake_reports in Drive. The original ZIP archives are left untouched. Large files are not stored on your laptop.

Inventories are intake aids, not training-ready metadata. Labels are candidates based on directory names; filename-based group hints are heuristic and marked for review. Do not split or train until rights, labels, and source/subject/session groups are checked against the dataset documentation.

If Colab reports insufficient temporary disk space, stop there and use a higher-storage cloud runtime. The notebook does not delete or overwrite existing Drive files.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import csv, hashlib, json, os, re, shutil, stat, zipfile
from collections import Counter
import requests
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

FOLDER_ID = "YOUR_GOOGLE_DRIVE_FOLDER_ID"
ARCHIVES = [
    {"key": "sdfvd2", "drive_id": "YOUR_SDFVD2_ARCHIVE_FILE_ID",
     "name": "SDFVD2.0_v1.zip", "drive_expected_bytes": 891163921,
     "download_expected_bytes": 891163921, "source_url": None},
    {"key": "comprehensive_v3", "drive_id": "YOUR_EXTERNAL_ARCHIVE_FILE_ID",
     "name": "Comprehensive_Deepfake_Dataset_v3.zip", "drive_expected_bytes": 4103008442,
     "download_name": "comprehensive_v3_inner.zip", "download_expected_bytes": 4103008442,
     "nested_member": "Comprehensive Deepfake Dataset Real and/deepfake_dataset.zip",
     "nested_expected_bytes": 4114687567, "source_url": None},
]
WORK_ROOT = Path("/content/deepfake_intake")
REPORT_ROOT = Path("/content/deepfake_intake_reports")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
drive = build("drive", "v3")

def file_metadata(file_id):
    return drive.files().get(
        fileId=file_id, fields="id,name,size,mimeType,parents,webViewLink,md5Checksum",
        supportsAllDrives=True,
    ).execute()

def download_drive_file(file_id, destination):
    request = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    digest, total = hashlib.sha256(), 0
    with open(destination, "w+b") as out:
        downloader = MediaIoBaseDownload(out, request, chunksize=32 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            end = out.tell()
            out.seek(total)
            digest.update(out.read(end - total))
            total = end
            out.seek(end)
            if status:
                print(f"Downloaded {total / (1024**3):.2f} GiB ({status.progress() * 100:.1f}%)")
    return total, digest.hexdigest()

def download_source(config, destination):
    if not config.get("source_url"):
        return download_drive_file(config["drive_id"], destination)
    digest, total = hashlib.sha256(), 0
    with requests.get(config["source_url"], stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with open(destination, "wb") as out:
            for chunk in response.iter_content(chunk_size=32 * 1024 * 1024):
                if chunk:
                    out.write(chunk)
                    digest.update(chunk)
                    total += len(chunk)
                    print(f"Downloaded {total / (1024**3):.2f} GiB")
    return total, digest.hexdigest()

def safe_member_path(destination, info):
    normalized = info.filename.replace("\\", "/")
    relative = PurePosixPath(normalized)
    if relative.is_absolute() or any(part in ("", ".", "..") for part in relative.parts):
        raise ValueError(f"Unsafe ZIP path: {info.filename!r}")
    mode = info.external_attr >> 16
    if stat.S_ISLNK(mode):
        raise ValueError(f"Refusing symbolic link in archive: {info.filename!r}")
    target = (destination / Path(*relative.parts)).resolve()
    if destination.resolve() not in target.parents:
        raise ValueError(f"ZIP path escapes extraction directory: {info.filename!r}")
    return target, relative

def label_candidate(parts):
    directories = {part.casefold() for part in parts[:-1]}
    real = any(part in {"real", "original", "authentic"} or part.endswith(("_real", "-real")) for part in directories)
    synthetic = any(part in {"fake", "deepfake", "synthetic", "forged"} or part.endswith(("_fake", "-fake")) for part in directories)
    if real and not synthetic:
        return "real", "directory_name"
    if synthetic and not real:
        return "synthetic", "directory_name"
    return "unknown", "ambiguous_or_not_in_path"

def group_hint(dataset_key, filename):
    if dataset_key != "sdfvd2":
        return "", "not_inferred", True
    stem = Path(filename).stem
    stem = re.sub(r"_aug_\d+$", "", stem, flags=re.IGNORECASE)
    token = re.sub(r"^(?:real|fake|deepfake|synthetic)_", "", stem, flags=re.IGNORECASE).casefold()
    if not token:
        return "", "not_inferred", True
    return hashlib.sha256(token.encode("utf-8")).hexdigest()[:20], "normalized_filename_candidate", True

def extract_and_inventory(config):
    metadata = file_metadata(config["drive_id"])
    if metadata.get("mimeType") != "application/zip":
        raise ValueError(f"{config['name']} is not a ZIP in Drive.")
    if FOLDER_ID not in metadata.get("parents", []):
        raise ValueError(f"{config['name']} is not in the designated Datasets folder.")
    actual_size = int(metadata.get("size", -1))
    if actual_size != config["drive_expected_bytes"]:
        raise ValueError(f"Drive size mismatch for {config['name']}: expected {config['drive_expected_bytes']}, got {actual_size}.")

    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = Path("/content") / config.get("download_name", config["name"])
    source_path = Path("/content") / config["name"] if config.get("nested_member") else archive_path
    extraction_root = WORK_ROOT / config["key"]
    if extraction_root.exists() and any(extraction_root.iterdir()):
        raise FileExistsError(f"{extraction_root} already has files; start a fresh Colab runtime to avoid overwriting.")
    free = shutil.disk_usage("/content").free
    download_size = config["download_expected_bytes"]
    if free < download_size + 1024**3:
        raise OSError(f"Insufficient Colab temporary space to download {config['name']} safely.")
    source = "Mendeley file URL" if config.get("source_url") else "Drive"
    print(f"Downloading {config['name']} from {source} into temporary Colab storage...")
    downloaded_size, source_sha256 = download_source(config, source_path)
    if downloaded_size != download_size:
        raise IOError(f"Downloaded {downloaded_size:,} bytes; expected {download_size:,}.")
    archive_sha256 = source_sha256
    if config.get("nested_member"):
        nested_size = config["nested_expected_bytes"]
        if shutil.disk_usage("/content").free < nested_size + 512 * 1024**2:
            raise OSError(f"Insufficient temporary space to unwrap {config['name']} safely; need {nested_size:,} additional bytes plus working margin.")
        nested_digest = hashlib.sha256()
        with zipfile.ZipFile(source_path, "r") as wrapper:
            members = [info for info in wrapper.infolist() if not info.is_dir()]
            matches = [info for info in members if info.filename == config["nested_member"]]
            if len(members) != 1 or len(matches) != 1 or matches[0].file_size != nested_size:
                raise ValueError("Drive wrapper ZIP contents do not match the expected nested archive metadata.")
            with wrapper.open(matches[0], "r") as inp, archive_path.open("wb") as out:
                while chunk := inp.read(8 * 1024 * 1024):
                    out.write(chunk)
                    nested_digest.update(chunk)
        source_path.unlink()
        archive_sha256 = nested_digest.hexdigest()
        print("Unwrapped the nested archive; wrapper CRC passed.")
    extraction_root.mkdir(parents=True, exist_ok=True)

    inventory_path = REPORT_ROOT / f"{config['key']}_inventory_{RUN_ID}.csv"
    extensions, labels, sample_labels, top_levels, auxiliary = Counter(), Counter(), Counter(), Counter(), Counter()
    total_bytes, count = 0, 0
    with zipfile.ZipFile(archive_path, "r") as archive:
        members = [info for info in archive.infolist() if not info.is_dir()]
        expanded = sum(info.file_size for info in members)
        free = shutil.disk_usage("/content").free
        if free < expanded + 1024**3:
            raise OSError(f"Insufficient Colab temporary space to extract {config['name']}: need about {expanded + 1024**3:,} bytes.")
        with inventory_path.open("w", newline="", encoding="utf-8") as csv_file:
            fields = ["dataset", "relative_path", "bytes", "extension", "label_candidate",
                      "label_basis", "auxiliary_entry", "group_candidate", "group_basis", "group_requires_manual_review"]
            writer = csv.DictWriter(csv_file, fieldnames=fields)
            writer.writeheader()
            for info in members:
                target, relative = safe_member_path(extraction_root, info)
                target.parent.mkdir(parents=True, exist_ok=True)
                # Reading each member fully also validates its ZIP CRC during extraction.
                with archive.open(info, "r") as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
                parts = relative.parts
                label, label_basis = label_candidate(parts)
                is_auxiliary = any(part == "__MACOSX" for part in parts) or relative.name.casefold() == ".ds_store" or Path(relative.name).suffix.casefold() not in {".jpg", ".jpeg", ".png", ".webp", ".mp4", ".mov", ".mkv", ".avi", ".webm"}
                group, group_basis, review = group_hint(config["key"], relative.name)
                extension = Path(relative.name).suffix.casefold() or "[no_extension]"
                writer.writerow({
                    "dataset": config["key"], "relative_path": relative.as_posix(),
                    "bytes": info.file_size, "extension": extension,
                    "label_candidate": label, "label_basis": label_basis, "auxiliary_entry": is_auxiliary,
                    "group_candidate": group, "group_basis": group_basis,
                    "group_requires_manual_review": review,
                })
                extensions[extension] += 1
                if is_auxiliary:
                    auxiliary["flagged"] += 1
                labels[label] += 1
                if not is_auxiliary:
                    sample_labels[label] += 1
                top_levels[parts[0]] += 1
                total_bytes += info.file_size
                count += 1
    archive_path.unlink()

    summary = {
        "dataset_key": config["key"], "archive_name": config["name"],
        "drive_file_id": config["drive_id"], "drive_wrapper_size_bytes": actual_size,
        "download_source_url": config.get("source_url"),
        "downloaded_size_bytes": downloaded_size, "archive_sha256": archive_sha256,
        "zip_crc_check": "all extracted members passed Python zipfile CRC checks",
        "file_count": count, "uncompressed_bytes": total_bytes,
        "extension_counts": dict(extensions), "label_candidate_counts_including_auxiliary": dict(labels),
        "label_candidate_counts_excluding_auxiliary": dict(sample_labels),
        "auxiliary_entries_flagged": auxiliary.get("flagged", 0),
        "top_level_path_counts": dict(top_levels),
        "extracted_to_ephemeral_colab_path": str(extraction_root),
        "inventory_report": inventory_path.name,
        "training_clearance": "not established by this inventory",
        "manual_review_required": True,
    }
    return summary, inventory_path

query = (
    f"name = 'intake_reports' and mimeType = 'application/vnd.google-apps.folder' "
    f"and '{FOLDER_ID}' in parents and trashed = false"
)
matches = drive.files().list(q=query, fields="files(id,name)", pageSize=10).execute().get("files", [])
if matches:
    report_folder_id = matches[0]["id"]
else:
    report_folder_id = drive.files().create(
        body={"name": "intake_reports", "mimeType": "application/vnd.google-apps.folder", "parents": [FOLDER_ID]},
        fields="id",
    ).execute()["id"]

def upload_report(path):
    mime = "application/json" if path.suffix == ".json" else "text/csv"
    media = MediaFileUpload(str(path), mimetype=mime, resumable=True)
    result = drive.files().create(
        body={"name": path.name, "parents": [report_folder_id]},
        media_body=media, fields="id,name,size,webViewLink,parents", supportsAllDrives=True,
    ).execute()
    print("Saved report:", result)
    return result

summaries, inventory_paths = [], []
for archive in ARCHIVES:
    print(f"\n=== {archive['name']} ===")
    summary, inventory = extract_and_inventory(archive)
    summaries.append(summary)
    inventory_paths.append(inventory)
    upload_report(inventory)
    print(json.dumps(summary, indent=2))

combined = {
    "run_id_utc": RUN_ID, "source_drive_folder_id": FOLDER_ID, "reports": summaries,
    "limitations": [
        "Directory labels are candidates only; compare them with source documentation.",
        "Filename-derived group candidates are heuristic and require manual review.",
        "Public availability and this inventory do not establish consent, rights, or training authorization.",
        "Do not create train/validation/test splits until source and subject/session groups are verified.",
    ],
}
summary_path = REPORT_ROOT / f"intake_summary_{RUN_ID}.json"
summary_path.write_text(json.dumps(combined, indent=2), encoding="utf-8")

upload_report(summary_path)

print("\nIntake complete. Original Drive ZIPs are untouched. Extracted media remains in Colab temporary storage.")
